<a href="https://colab.research.google.com/github/Alafiade/Digital-Path/blob/main/OPEN_SLIDE_LIBRARY_FOR_WHOLE_SLIDE_IMAGES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

INSTALLING LIBRARIES

In [ ]:
!pip install openslide-python

In [ ]:
! apt-get install openslide-tools

In [ ]:
import openslide

In [ ]:
from PIL import Image
import numpy as np
from matplotlib import pyplot as plt

DOWNLOADING WSI DATA FROM HUGGING FACE

In [ ]:
from huggingface_hub import hf_hub_download

slide = hf_hub_download(
    "rendeirolab/lazyslide-data",
    "GTEX-1117F-0526.svs",
    repo_type = "dataset",
    cache_dir = "."

)

In [ ]:
#Wrapping our data in Openslide object to handle massive file sizes
slide_object = openslide.OpenSlide(slide)

# Viewing the WSIs properties
slide_props = slide_object.properties

PRINTING THE PROPERTIES OF OUR WHOLE SLIDE IMAGES

In [ ]:
slide_props

In [ ]:
print("Vendor is:",slide_props['openslide.vendor'])

In [ ]:
print("Pixel  size of X  in um is:",slide_props["openslide.mpp-x"])
print("Pixel size of Y in um is:", slide_props["openslide.mpp-y"])

In [ ]:
#Displaying the native resolution of the Whole Slide Image
slide_dims = slide_object.dimensions
print(slide_dims)

In [ ]:
slide_thumb_256 = slide_object.get_thumbnail(size=(256,256))
slide_thumb_256.show()

In [ ]:
slide_thumb_256_np = np.array(slide_thumb_256)
plt.figure(figsize=(6,6))
plt.imshow(slide_thumb_256_np)


In [ ]:
#Whole Slide Images have a pyramidial structure
# slide.level_dimensions displays the pyramidial levels

dims = slide_object.level_dimensions
dims

In [ ]:
num_levels = len(dims)
print("The levels of WSI images are:",num_levels)

In [ ]:
# Assessing the Levels of the WSI images using indexing

level_3_dims = dims[2]
level_3_dims

In [ ]:
# Scaling factor of the WSI image
# This code displays how much each level in the WSI is scaled down

factors = slide_object.level_downsamples
print('Each level is downsampled by an amount of: ',factors)

In [ ]:
# This reads the topleft image of the third level in the WSI

level_3_dims = dims[2]
level_3_img = slide_object.read_region((0,0),2, level_3_dims)

In [ ]:
#Converting the PIL image from RGBA to RGB for analysis
level_3_img_RGB = level_3_img.convert('RGB')
level_3_img_RGB.show()

In [ ]:
level_3_img_np = np.array(level_3_img_RGB)
plt.imshow(level_3_img_np)

Due to WSI's large image file, the Images are downsampled  relative to the highest resolution (level0)

In [ ]:
#This downsamples the original image resolution to a factor of 32
SCALE_FACTOR = 32
#This picks the best level closest to a scaled down factor of 32
best_level = slide_object.get_best_level_for_downsample(SCALE_FACTOR)

In [ ]:
best_level

In [ ]:
 from openslide.deepzoom import DeepZoomGenerator

In [ ]:
#This splits our WSI into tiles of 256x256pixels
tiles = DeepZoomGenerator(slide_object, tile_size=256, overlap=0, limit_bounds = False)

In [ ]:
print("The number of levels in our tiles are:", tiles.level_count)

In [ ]:
print("The dimensions of data in each level are:", tiles.level_dimensions)

In [ ]:
print("Toal number of tiles:",tiles.tile_count)

In [ ]:
#Inspecting the Tile shape at each level
level_num = 11
print("Tiles shape at level", level_num,"is:", tiles.level_tiles[level_num])
print("This means there are", tiles.level_tiles[level_num][0],"tiles.level_tiles")

In [ ]:
tile_dims = tiles.get_tile_dimensions(11,(0,3))
tile_dims

In [ ]:
#Counting the number of tiles in the level 16
level_num = 15
tile_count_in_large_image = tiles.level_tiles[level_num]
print(tile_count_in_large_image)

In [ ]:
#Displaying the tile dimensions

tile_dims_in_large_image = tiles.get_tile_dimensions(15,(77,77))
tile_dims_in_large_image

In [ ]:
# Visualizing/Displaying a single tile from their dimensions

single_tile = tiles.get_tile(15,(60,15))
single_tile_RGB = single_tile.convert('RGB')
single_tile_np = np.array(single_tile_RGB)
plt.imshow(single_tile_np)
plt.show()

In [ ]:
# Saving Tiles level to hard disk for preprocessing
cols, rows = tiles.level_tiles[15]

import os

tile_dir  = '/content/WSI level 0 sllide  tiles'
for row in range(rows):
  for col in range(cols):
    tile_name = os.path.join(tile_dir,"%d_%d"% (col,row))
    print("Now saving tile with tile:", tile_name)
    temp_tile = tiles.get_tile(15, (col, row))
    temp_tile_RGB = temp_tile.convert('RGB')
    temp_tile_np = np.array(temp_tile_RGB)
    plt.imsave(tile_name + ".png", temp_tile_np)

